<h2>1. Introdução</h2>
Este projeto foi motivado pelo interesse em investigar padrões e mitos populares sobre a indústria cinematográfica, utilizando dados históricos do IMDB combinados com metadados enriquecidos via APIs do TMDB e OMDB. Através da coleta e cruzamento desses dados, buscamos testar hipóteses recorrentes sobre duração, gênero, idioma, orçamento/lucratividade e recepção crítica vs. popular.

<h3>1.1 Objetivo Geral</h3>
<ul><li>Investigar tendências e mitos populares sobre cinema a partir de dados históricos de duração, gênero, idioma, orçamento, receita e avaliação crítica vs. de público.</li></ul>

<h3>1.2 Objetivos Específicos</h3>
<ul>
<li>Comparar a duração de filmes de franquia contra filmes isolados</li>
<li>Analisar a evolução histórica do gênero Romance em termos de volume de produção e popularidade</li>
<li>Verificar a evolução da predominância de filmes em língua inglesa ao longo do tempo</li>
<li>Investigar a relação entre orçamento de produção e retorno financeiro/lucratividade</li>
<li>Comparar a divergência entre nota de crítica e nota de público entre diferentes gêneros</li>
</ul>

<h3>1.3 Perguntas a Serem Respondidas</h3>
<ol>
<li>Franquias de cinema realmente exigem mais tempo do público do que filmes originais? (Murilo)</li>
<li>O gênero Romance está morrendo? (Gabriel)</li>
<li>O domínio de filmes em língua inglesa está diminuindo? (Icaro)</li>
<li>Filmes baratos podem ser mais lucrativos que filmes caros? (Gabriel)</li>
<li>A divergência entre crítica e público é maior em algum gênero específico? (Kawane)</li>
</ol>

<h3>1.4 Bibliotecas utilizadas:</h3>
As bibliotecas utilizadas estão contidas no arquivo requirements.txt no diretório do presente trabalho.
>pip install -r requirements.txt

<h2>2. Dados Utilizados</h2>

Os dados utilizados correspondem a filmes registrados na base pública do IMDB, filtrados para conter apenas títulos com no mínimo 10.000 avaliações de usuários, garantindo relevância estatística mínima por filme. A base inicial foi posteriormente enriquecida com metadados obtidos através das APIs do TMDB (The Movie Database) e da OMDB (Open Movie Database), permitindo a inclusão de variáveis não presentes na base original do IMDB, como idioma original, orçamento, receita, indicador de franquia e nota de crítica especializada.

O uso de múltiplas fontes foi necessário porque nenhuma delas isoladamente contém todas as variáveis necessárias para responder às cinco perguntas de pesquisa do grupo (duração e franquia, evolução de gêneros, idioma, orçamento/receita, e divergência crítica vs. público).

<h3>2.1 Coleta de Dados</h3>

A coleta de dados foi realizada em três etapas sequenciais:

<b>1. Base do IMDB:</b> o IMDB disponibiliza publicamente arquivos de dados em formato .tsv.gz, atualizados periodicamente, sem necessidade de autenticação ou chave de API. Foram utilizados dois arquivos:
<ul>
<li><code>title.basics.tsv.gz</code>: contém informações básicas de cada título (tipo, nome, ano, duração, gêneros).</li>
<li><code>title.ratings.tsv.gz</code>: contém a nota média e o número de avaliações de usuários para cada título.</li>
</ul>
Os dois arquivos foram unidos (merge) pela chave <code>tconst</code> (identificador único do IMDB), mantendo apenas títulos do tipo "movie" com no mínimo 10.000 votos.

<b>2. Enriquecimento via TMDB API:</b> o TMDB disponibiliza uma API pública mediante chave de autenticação gratuita. Para cada filme da base (identificado pelo <code>tconst</code>, compatível com o ID do IMDB), foi feita uma requisição ao endpoint <code>/movie/{imdb_id}</code>, extraindo:
<ul>
<li>Idioma original de produção (<code>original_language</code>)</li>
<li>Indicador de pertencimento a uma franquia/coleção (<code>is_franchise</code>), obtido a partir do campo <code>belongs_to_collection</code></li>
<li>Orçamento de produção (<code>budget</code>)</li>
<li>Receita de bilheteria (<code>revenue</code>)</li>
</ul>

<b>3. Enriquecimento via OMDB API:</b> a partir da base já enriquecida com o TMDB, foi feita uma segunda rodada de requisições à API da OMDB, também usando o <code>tconst</code> como identificador, para obter a nota de crítica especializada (<code>Metascore</code>), utilizada na comparação entre avaliação de crítica e avaliação de público.

<h3>2.2 Scripts de Coleta</h3>

Em relação à base do IMDB:

In [ ]:
import pandas as pd

df_ratings = pd.read_csv('../data/title.ratings.tsv.gz', sep='\t')
df_ratings = df_ratings[df_ratings['numVotes'] >= 10000]

colunas_basics = ['tconst', 'titleType', 'primaryTitle', 'startYear', 'runtimeMinutes', 'genres']
df_basics = pd.read_csv('../data/title.basics.tsv.gz', sep='\t', usecols=colunas_basics, low_memory=False)
df_basics = df_basics[df_basics['titleType'] == 'movie']

base_projeto = pd.merge(df_basics, df_ratings, on='tconst', how='inner')
base_projeto.to_csv('../data/base_projeto.csv', index=False)

Em relação ao TMDB:



In [ ]:
import pandas as pd
import requests
import time
import os
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv('../.env')
API_KEY = os.getenv('TMDB_API_KEY')

df_base = pd.read_csv('../data/base_projeto.csv')

original_language, is_franchise, budget, revenue = [], [], [], []

for imdb_id in tqdm(df_base['tconst']):
    url_tmdb = f"https://api.themoviedb.org/3/movie/{imdb_id}?api_key={API_KEY}"
    try:
        response_tmdb = requests.get(url_tmdb)
        if response_tmdb.status_code == 200:
            data = response_tmdb.json()
            original_language.append(data.get('original_language'))
            budget.append(data.get('budget', 0))
            revenue.append(data.get('revenue', 0))
            is_franchise.append(bool(data.get('belongs_to_collection')))
        else:
            original_language.append(None)
            budget.append(None)
            revenue.append(None)
            is_franchise.append(False)
    except Exception:
        original_language.append(None)
        budget.append(None)
        revenue.append(None)
        is_franchise.append(False)
    time.sleep(0.02)

df_base['original_language'] = original_language
df_base['is_franchise'] = is_franchise
df_base['budget'] = budget
df_base['revenue'] = revenue
df_base.to_csv('../data/df_tmdb.csv', index=False)

Em relação à OMDB:

In [ ]:
import pandas as pd
import requests
import time
import os
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv('../.env')
API_KEY = os.getenv('OMDB_API_KEY')

df = pd.read_csv('../data/df_tmdb.csv')
metascore_critica = []

for index, row in tqdm(df.iterrows(), total=len(df)):
    imdb_id = row['tconst']
    url_omdb = f"http://www.omdbapi.com/?i={imdb_id}&apikey={API_KEY}"
    try:
        response = requests.get(url_omdb)
        if response.status_code == 200:
            data = response.json()
            score = data.get('Metascore')
            metascore_critica.append(int(score) if score and score != "N/A" else -1)
        else:
            metascore_critica.append(None)
        time.sleep(0.03)
    except Exception:
        metascore_critica.append(None)

df['metascore_critica'] = metascore_critica
df.to_csv('../data/df_omdb.csv', index=False)

<i>Observação: as células de coleta via API acima não são reexecutadas neste notebook, pois a coleta completa via TMDB consome aproximadamente 1h16min e a via OMDB aproximadamente 45min, além de gastar cota de requisições das APIs. Os resultados já coletados são carregados diretamente dos arquivos CSV salvos, exibidos na seção 2.3.</i>

<h3>2.3 Armazenamento dos Dados</h3>

Os dados foram armazenados de forma incremental, em três arquivos CSV, permitindo visualizar o processo de enriquecimento em cada etapa:

In [ ]:
# 1. Base original, gerada a partir dos arquivos do IMDB
df_base = pd.read_csv('../data/base_projeto.csv')
print(f"Filmes na base original: {len(df_base)}")
df_base.head()

In [ ]:
# 2. Base enriquecida com dados do TMDB (idioma, franquia, orçamento, receita)
df_tmdb = pd.read_csv('../data/df_tmdb.csv')
print(f"Colunas adicionadas: original_language, is_franchise, budget, revenue")
df_tmdb.head()

In [ ]:
# 3. Base enriquecida com dados da OMDB (nota de crítica) -- base final utilizada nas análises
df_omdb = pd.read_csv('../data/df_omdb.csv')
print(f"Coluna adicionada: metascore_critica")
df_omdb.head()

O arquivo <code>df_omdb.csv</code> constitui a base final e completa, utilizada por todos os integrantes do grupo em suas respectivas análises.

<h3>2.4 Descrição do dataset</h3>

As colunas do dataset final (<code>df_omdb.csv</code>) são:
<ul>
<li><b>tconst:</b> identificador único do título no IMDB</li>
<li><b>titleType:</b> tipo de título (filtrado para "movie")</li>
<li><b>primaryTitle:</b> título principal do filme</li>
<li><b>startYear:</b> ano de lançamento</li>
<li><b>runtimeMinutes:</b> duração em minutos</li>
<li><b>genres:</b> gênero(s) do filme</li>
<li><b>averageRating:</b> nota média de avaliação do público no IMDB</li>
<li><b>numVotes:</b> número de avaliações recebidas no IMDB</li>
<li><b>original_language:</b> idioma original de produção (obtido via TMDB)</li>
<li><b>is_franchise:</b> indica se o filme pertence a uma franquia/coleção (obtido via TMDB)</li>
<li><b>budget:</b> orçamento de produção em dólares (obtido via TMDB)</li>
<li><b>revenue:</b> receita de bilheteria em dólares (obtido via TMDB)</li>
<li><b>metascore_critica:</b> nota de crítica especializada, de 0 a 100 (obtido via OMDB); valor -1 indica ausência de nota disponível</li>
</ul>

<h3>2.5 Exploração Inicial dos Dados</h3>

Importando as bibliotecas necessárias:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Remoção apenas para fins estéticos do notebook
import warnings
warnings.filterwarnings('ignore')

Carregar os dados:

In [ ]:
df = pd.read_csv('../data/df_omdb.csv')

Visualizar as primeiras linhas do dataset:

In [ ]:
df.head()

Verificar o tipo de dados e se há valores ausentes:


In [ ]:
df.info()

Estatísticas descritivas iniciais:

In [ ]:
df.describe()

<h2>3. Pré-processamento</h2>

<h3>3.1 Limpeza e Transformação dos Dados</h3>

Diferentemente de conjuntos de dados obtidos por raspagem, os dados coletados via arquivos oficiais do IMDB e via APIs do TMDB e OMDB já chegaram estruturados e com tipagem consistente, não sendo necessário um processo de limpeza textual (como padronização de nomes ou remoção de caracteres inválidos). Ainda assim, algumas decisões de tratamento foram tomadas durante e após a coleta, detalhadas na seção 3.2.

Antes de iniciar as análises individuais, foi realizada uma verificação de duplicidade e consistência de tipos:

In [ ]:
# Verificação de duplicatas por identificador único
print(f"Duplicatas por tconst: {df['tconst'].duplicated().sum()}")

# Verificação de valores ausentes por coluna
df.isnull().sum()

<h3>3.2 Explicando Decisões Tomadas</h3>

Ao longo da construção da base de dados, foram tomadas as seguintes decisões de pré-processamento:

<ul>
<li><b>Filtro de relevância estatística:</b> foram mantidos apenas filmes com no mínimo 10.000 avaliações de usuários no IMDB (<code>numVotes >= 10000</code>), evitando que títulos pouco avaliados distorçam métricas de popularidade e nota média.</li>

<li><b>Junção interna (inner join) entre IMDB basics e ratings:</b> apenas filmes presentes em ambos os arquivos do IMDB foram mantidos, descartando títulos sem avaliação registrada.</li>

<li><b>Tratamento de falhas de requisição (TMDB e OMDB):</b> quando uma requisição à API falhava (erro de conexão ou status diferente de 200), os campos correspondentes foram preenchidos com <code>None</code>, distinguindo-os de casos em que a API respondeu com sucesso mas sem o dado (por exemplo, <code>budget</code> e <code>revenue</code> retornados como 0 pelo próprio TMDB). Essa distinção deve ser considerada nas análises que envolvem orçamento e receita, já que um valor 0 pode representar tanto ausência real de informação quanto uma resposta válida da API.</li>

<li><b>Valor sentinela para nota de crítica ausente:</b> filmes sem Metascore disponível na OMDB foram marcados com o valor -1 (em vez de nulo), para diferenciar explicitamente "sem nota de crítica disponível" de possíveis erros de requisição (marcados como <code>None</code>). Esse valor foi filtrado antes de qualquer cálculo estatístico envolvendo a coluna <code>metascore_critica</code>.</li>
</ul>

<h2>4. Perguntas e Análises</h2>

<h3>4.1 Franquias de cinema realmente exigem mais tempo do público do que filmes originais?</h3>

<h3>🎯 Motivação e Hipóteses</h3>
Existe uma percepção popular de que filmes de franquia e grandes sagas de Hollywood estão ficando cada vez mais longos e cansativos. Esta análise investiga essa hipótese a partir de três perspectivas:
<ol>
<li><b>Mito vs. Realidade:</b> Filmes de franquia têm duração média estatisticamente maior do que filmes isolados (<i>Standalone</i>)?</li>
<li><b>O Efeito Acumulativo:</b> À medida que uma saga progride (Filme 1, Filme 2, Filme 3+), a duração aumenta de forma contínua?</li>
<li><b>Evolução Temporal:</b> Essa diferença de duração entre franquias e filmes isolados é um fenômeno recente ou sempre existiu?</li>
</ol>

<h3>4.1.1 Sub-hipótese 1: Mito vs. Realidade</h3>
<b>Pergunta:</b> Filmes de franquia têm duração média/mediana superior à de filmes isolados (<i>Standalone</i>)?

Nesta etapa, comparamos a distribuição da duração (<code>runtimeMinutes</code>) entre os dois grupos:
<ul>
<li><b>Filmes Isolados (<code>is_franchise == False</code>):</b> Produções sem sequências registradas.</li>
<li><b>Filmes de Franquia (<code>is_franchise == True</code>):</b> Produções vinculadas a coleções/sagas.</li>
</ul>

In [ ]:
codigo aqui

Análise:

<h3>4.1.2 Sub-hipótese 2: O Efeito Acumulativo</h3>
<b>Pergunta:</b> À medida que uma saga progride (Filme 1, Filme 2, Filme 3+), a duração aumenta de forma contínua?

Para responder essa pergunta, foi necessário identificar a posição de cada filme dentro de sua franquia (1º, 2º, 3º...), informação obtida via API do TMDB (campo <code>belongs_to_collection</code> e a lista ordenada de <code>parts</code> de cada coleção, ordenadas por <code>release_date</code>). O resultado foi salvo em <code>df_franquias_com_posicao.csv</code>.

In [ ]:
codigo aqui

Análise:

<h3>4.1.3 Sub-hipótese 3: Evolução Temporal</h3>
<b>Pergunta:</b> A diferença de duração entre franquias e filmes isolados é um fenômeno recente ou sempre existiu?

Aqui investigamos se o "gap" de duração entre os dois grupos se abriu ao longo do tempo, usando uma regressão linear com termo de interação (<code>runtimeMinutes ~ startYear * is_franchise</code>), que testa formalmente se a taxa de crescimento da duração ao longo dos anos difere entre franquias e isolados.

In [ ]:
codigo aqui

Análise: